In [1]:
# ============================================================
# APPEEARS SETUP
# ============================================================

import os
import time
import getpass
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests


APPEEARS_API = (
    "https://appeears.earthdatacloud.nasa.gov/api/"
)

APPEEARS_PRODUCT = "SPL4SMGP.008"
APPEEARS_LAYER = "sm_surface"

APPEEARS_DIR = Path(
    "Data/Global/smap_appeears"
)

APPEEARS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "AppEEARS API:",
    APPEEARS_API,
)

print(
    "Product:",
    APPEEARS_PRODUCT,
)

print(
    "Layer:",
    APPEEARS_LAYER,
)

print(
    "Output:",
    APPEEARS_DIR,
)

AppEEARS API: https://appeears.earthdatacloud.nasa.gov/api/
Product: SPL4SMGP.008
Layer: sm_surface
Output: Data/Global/smap_appeears


In [2]:
# ============================================================
# APPEEARS EARTHDATA LOGIN
# ============================================================

appeears_user = getpass.getpass(
    "NASA Earthdata username: "
)

appeears_password = getpass.getpass(
    "NASA Earthdata password: "
)

login_response = requests.post(
    APPEEARS_API + "login",
    auth=(
        appeears_user,
        appeears_password,
    ),
    timeout=60,
)

print(
    "HTTP status:",
    login_response.status_code,
)

login_response.raise_for_status()

token_response = (
    login_response
    .json()
)

appeears_token = (
    token_response["token"]
)

appeears_headers = {
    "Authorization":
        f"Bearer {appeears_token}",
}

del appeears_user
del appeears_password

print(
    "AppEEARS authentication successful."
)

HTTP status: 200
AppEEARS authentication successful.


In [4]:
# ============================================================
# CELL C — INSPECT ACTUAL APPEEARS SPL4SMGP.008 LAYERS
# ============================================================

products_response = requests.get(
    APPEEARS_API + "product",
    timeout=60,
)

products_response.raise_for_status()

products = products_response.json()

product_lookup = {
    p["ProductAndVersion"]: p
    for p in products
}

if APPEEARS_PRODUCT not in product_lookup:

    matching = [
        p["ProductAndVersion"]
        for p in products
        if "SPL4SMGP" in
        p["ProductAndVersion"]
    ]

    raise RuntimeError(
        "SPL4SMGP.008 was not found in AppEEARS.\n"
        f"Available SPL4SMGP products: {matching}"
    )

print(
    "Confirmed product:",
    APPEEARS_PRODUCT,
)

# ------------------------------------------------------------
# Retrieve actual layer metadata
# ------------------------------------------------------------

layer_response = requests.get(
    APPEEARS_API
    + "product/"
    + APPEEARS_PRODUCT,
    timeout=60,
)

layer_response.raise_for_status()

layers = layer_response.json()

print(
    "\nResponse type:",
    type(layers),
)

print(
    "\nAvailable layer identifiers:"
)

if isinstance(layers, dict):

    for key in layers.keys():
        print(
            " -",
            key,
        )

else:

    print(
        layers
    )


# ------------------------------------------------------------
# Print detailed metadata for every layer
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "DETAILED LAYER METADATA"
)

print(
    "============================================================"
)

if isinstance(layers, dict):

    for key, metadata in layers.items():

        print(
            "\nLAYER:",
            key,
        )

        print(
            metadata
        )

else:

    for item in layers:

        print(
            "\nITEM:"
        )

        print(
            item
        )

Confirmed product: SPL4SMGP.008

Response type: <class 'dict'>

Available layer identifiers:
 - Geophysical_Data_baseflow_flux
 - Geophysical_Data_depth_to_water_table_from_surface
 - Geophysical_Data_free_surface_water_on_peat_flux
 - Geophysical_Data_heat_flux_ground
 - Geophysical_Data_heat_flux_latent
 - Geophysical_Data_heat_flux_sensible
 - Geophysical_Data_height_lowatmmodlay
 - Geophysical_Data_land_evapotranspiration_flux
 - Geophysical_Data_land_fraction_saturated
 - Geophysical_Data_land_fraction_snow_covered
 - Geophysical_Data_land_fraction_unsaturated
 - Geophysical_Data_land_fraction_wilting
 - Geophysical_Data_leaf_area_index
 - Geophysical_Data_mwrtm_vegopacity
 - Geophysical_Data_net_downward_longwave_flux
 - Geophysical_Data_net_downward_shortwave_flux
 - Geophysical_Data_overland_runoff_flux
 - Geophysical_Data_precipitation_total_surface_flux
 - Geophysical_Data_radiation_longwave_absorbed_flux
 - Geophysical_Data_radiation_shortwave_downward_flux
 - Geophysical_Da

In [7]:
# ============================================================
# CELL C2 — FIND THE SURFACE SOIL MOISTURE LAYER
# ============================================================

def metadata_to_text(obj):
    """
    Recursively convert AppEEARS layer metadata to searchable text.
    """
    if isinstance(obj, dict):
        return " ".join(
            [
                str(k)
                + " "
                + metadata_to_text(v)
                for k, v in obj.items()
            ]
        )

    if isinstance(obj, (list, tuple)):
        return " ".join(
            metadata_to_text(v)
            for v in obj
        )

    return str(obj)


layer_candidates = []

if isinstance(layers, dict):

    for layer_name, metadata in layers.items():

        text = (
            layer_name
            + " "
            + metadata_to_text(
                metadata
            )
        ).lower()

        score = 0

        if "soil" in text:
            score += 3

        if "moisture" in text:
            score += 3

        if "surface" in text:
            score += 3

        if "0-5" in text:
            score += 2

        if "volume fraction" in text:
            score += 1

        if score > 0:

            layer_candidates.append(
                (
                    score,
                    layer_name,
                    metadata,
                )
            )


layer_candidates = sorted(
    layer_candidates,
    key=lambda x: (
        -x[0],
        x[1],
    ),
)


print(
    "Possible surface-soil-moisture layers:"
)

for score, name, metadata in (
    layer_candidates
):

    print(
        "\nScore:",
        score,
        "| Layer:",
        name,
    )

    print(
        metadata
    )


if not layer_candidates:

    raise RuntimeError(
        "Could not identify a soil-moisture "
        "surface layer from the AppEEARS metadata."
    )

# ------------------------------------------------------------
# Use the highest-scoring candidate
# ------------------------------------------------------------

# ============================================================
# SET THE VERIFIED APPEEARS SMAP LAYER
# ============================================================

APPEEARS_LAYER = (
    "Geophysical_Data_sm_surface"
)

print(
    "Using AppEEARS layer:",
    APPEEARS_LAYER,
)

Possible surface-soil-moisture layers:

Score: 11 | Layer: Geophysical_Data_sm_surface
{'AddOffset': '', 'Available': True, 'DataType': 'float32', 'Description': 'Top layer soil moisture (0-5 cm)', 'Dimensions': ['time', 'y', 'x'], 'FillValue': -9999.0, 'FillValueAll': [-9999.0], 'Group': '', 'Info': {'group_name': 'Geophysical_Data', 'layer_name': 'sm_surface'}, 'IsQA': False, 'Layer': 'Geophysical_Data_sm_surface', 'OrigDataType': 'float32', 'OrigValidMax': 0.8999999762, 'OrigValidMin': 0.0, 'QualityLayers': '', 'QualityProductAndVersion': '', 'ScaleFactor': '', 'Units': 'm^3/m^3', 'ValidMax': 0.8999999762, 'ValidMin': 0.0, 'XSize': 3856, 'YSize': 1624}

Score: 8 | Layer: Geophysical_Data_sm_surface_wetness
{'AddOffset': '', 'Available': True, 'DataType': 'float32', 'Description': 'Top layer soil wetness (0-5 cm; wetness units). Soil wetness units (dimensionless) vary between 0 and 1, indicating relative saturation between completely dry conditions and completely saturated conditions

In [9]:
# ============================================================
# APPEEARS PILOT STATIONS
# ============================================================

# Load station metadata directly if it is not already in memory.
if "stations" not in globals():

    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

# Make sure station_key exists.
if "station_key" not in stations.columns:

    stations["network"] = (
        stations["network"]
        .astype(str)
    )

    stations["station_id"] = (
        stations["station_id"]
        .astype(str)
    )

    stations["station_key"] = (
        stations["network"]
        + "::"
        + stations["station_id"]
    )

# ------------------------------------------------------------
# Select 10 stations for the pilot
# ------------------------------------------------------------

PILOT_N_STATIONS = 10

pilot_stations = (
    stations[
        [
            "station_key",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates(
        "station_key"
    )
    .sort_values(
        "station_key"
    )
    .head(
        PILOT_N_STATIONS
    )
    .reset_index(
        drop=True
    )
)

pilot_stations[
    "appeears_id"
] = np.arange(
    len(pilot_stations)
).astype(
    str
)

print(
    "Pilot stations:",
    len(pilot_stations)
)

display(
    pilot_stations
)

Pilot stations: 10


,station_key,latitude,longitude,appeears_id
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,13.53250,2.66040,0
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,9.79506,1.70994,1
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,9.78986,1.70994,2
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,9.74530,1.60530,3
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,9.74407,1.60580,4
5,AMMA-CATCH::AMMA-CATCH__Tondikiboro,13.54800,2.69560,5
6,AMMA-CATCH::AMMA-CATCH__Wankama,13.64560,2.63200,6
7,ARM::ARM__Anthony,37.21340,-98.09690,7
8,ARM::ARM__Ashton,37.13300,-97.26600,8
9,ARM::ARM__Byron,36.88100,-98.28500,9


In [17]:
# ============================================================
# APPEEARS PILOT TASK
# 10 STATIONS × 7 DAYS × SPL4SMGP.008
# ============================================================

PILOT_START = pd.Timestamp(
    "2015-04-01"
)

PILOT_END = pd.Timestamp(
    "2015-04-07"
)

# ------------------------------------------------------------
# Build AppEEARS point coordinates
# ------------------------------------------------------------

appeears_coordinates = []

for _, row in (
    pilot_stations.iterrows()
):

    appeears_coordinates.append({

        "id":
            str(
                row[
                    "appeears_id"
                ]
            ),

        "latitude":
            float(
                row[
                    "latitude"
                ]
            ),

        "longitude":
            float(
                row[
                    "longitude"
                ]
            ),

        "category":
            str(
                row[
                    "station_key"
                ]
            ),
    })


# ------------------------------------------------------------
# Build task
# ------------------------------------------------------------

appeears_task = {

    "task_type":
        "point",

    "task_name":
        "SMAP_L4_V008_pilot",

    "params": {

        "dates": [
            {
                "startDate":
                    PILOT_START.strftime(
                        "%m-%d-%Y"
                    ),

                "endDate":
                    PILOT_END.strftime(
                        "%m-%d-%Y"
                    ),
            }
        ],

        "layers": [
            {
                "product":
                    APPEEARS_PRODUCT,

                "layer":
                    APPEEARS_LAYER,
            }
        ],

        "coordinates":
            appeears_coordinates,
    },
}


print(
    "Submitting AppEEARS pilot..."
)

task_response = requests.post(
    APPEEARS_API + "task",
    json=appeears_task,
    headers=appeears_headers,
    timeout=60,
)

print(
    "HTTP status:",
    task_response.status_code,
)

print(
    task_response.text
)

task_response.raise_for_status()

task_json = (
    task_response.json()
)

appeears_task_id = (
    task_json["task_id"]
)

print(
    "\nAppEEARS task ID:",
    appeears_task_id,
)

Submitting AppEEARS pilot...
HTTP status: 202
{"task_id": "b4332185-4a3c-4936-b4f0-7560019ecccd", "status": "pending"}

AppEEARS task ID: b4332185-4a3c-4936-b4f0-7560019ecccd


In [11]:
# ============================================================
# WAIT FOR APPEEARS TASK
# ============================================================

POLL_SECONDS = 20

while True:

    status_response = requests.get(
        APPEEARS_API
        + "task/"
        + appeears_task_id,
        headers=appeears_headers,
        timeout=60,
    )

    status_response.raise_for_status()

    status_json = (
        status_response
        .json()
    )

    status = (
        status_json[
            "status"
        ]
    )

    print(
        time.strftime("%H:%M:%S"),
        status,
    )

    if status in {
        "done",
        "failed",
        "error",
    }:
        break

    time.sleep(
        POLL_SECONDS
    )

if status != "done":

    raise RuntimeError(
        "AppEEARS task did not finish "
        f"successfully:\n{status_json}"
    )

print(
    "\nAppEEARS pilot finished."
)

09:16:27 pending
09:16:48 processing
09:17:09 processing
09:17:30 processing
09:17:51 processing
09:18:12 processing
09:18:34 processing
09:18:54 processing
09:19:16 processing
09:19:37 done

AppEEARS pilot finished.


In [12]:
# ============================================================
# APPEEARS BUNDLE + DOWNLOAD
# ============================================================

bundle_response = requests.get(
    APPEEARS_API
    + "bundle/"
    + appeears_task_id,
    headers=appeears_headers,
    timeout=60,
)

bundle_response.raise_for_status()

bundle = (
    bundle_response
    .json()
)

print(
    "Bundle files:"
)

for item in bundle.get(
    "files",
    []
):

    print(
        item[
            "file_name"
        ],
        "|",
        item[
            "file_type"
        ],
        "|",
        item[
            "file_size"
        ],
        "bytes"
    )


# ------------------------------------------------------------
# Download all bundle files
# ------------------------------------------------------------

pilot_dir = (
    APPEEARS_DIR
    / "pilot"
)

pilot_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for item in bundle[
    "files"
]:

    file_id = item[
        "file_id"
    ]

    file_name = item[
        "file_name"
    ]

    download_url = (
        APPEEARS_API
        + "bundle/"
        + appeears_task_id
        + "/"
        + file_id
    )

    output_path = (
        pilot_dir
        / file_name
    )

    print(
        "\nDownloading:",
        file_name,
    )

    response = requests.get(
        download_url,
        headers=appeears_headers,
        stream=True,
        timeout=300,
    )

    response.raise_for_status()

    with open(
        output_path,
        "wb",
    ) as fh:

        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):

            if chunk:
                fh.write(
                    chunk
                )

    print(
        "Saved:",
        output_path,
    )

print(
    "\nPilot files saved to:",
    pilot_dir,
)

Bundle files:
SMAP-L4-V008-pilot-SPL4SMGP-008-results.csv | csv | 69411 bytes
SMAP-L4-V008-pilot-granule-list.txt | txt | 8176 bytes
SMAP-L4-V008-pilot-request.json | json | 1972 bytes
SMAP-L4-V008-pilot-SPL4SMGP-008-metadata.xml | xml | 20500 bytes
README.md | txt | 28997 bytes

Downloading: SMAP-L4-V008-pilot-SPL4SMGP-008-results.csv
Saved: Data/Global/smap_appeears/pilot/SMAP-L4-V008-pilot-SPL4SMGP-008-results.csv

Downloading: SMAP-L4-V008-pilot-granule-list.txt
Saved: Data/Global/smap_appeears/pilot/SMAP-L4-V008-pilot-granule-list.txt

Downloading: SMAP-L4-V008-pilot-request.json
Saved: Data/Global/smap_appeears/pilot/SMAP-L4-V008-pilot-request.json

Downloading: SMAP-L4-V008-pilot-SPL4SMGP-008-metadata.xml
Saved: Data/Global/smap_appeears/pilot/SMAP-L4-V008-pilot-SPL4SMGP-008-metadata.xml

Downloading: README.md
Saved: Data/Global/smap_appeears/pilot/README.md

Pilot files saved to: Data/Global/smap_appeears/pilot


In [13]:
# ============================================================
# INSPECT APPEEARS SMAP RESULT
# ============================================================

result_csvs = sorted(
    pilot_dir.glob(
        "*results.csv"
    )
)

print(
    "Result CSVs:"
)

for path in result_csvs:
    print(
        path.name
    )

if not result_csvs:

    raise RuntimeError(
        "No AppEEARS result CSV was found."
    )

pilot_csv = result_csvs[0]

pilot_df = pd.read_csv(
    pilot_csv
)

print(
    "\nResult shape:",
    pilot_df.shape
)

print(
    "\nColumns:"
)

print(
    pilot_df.columns.tolist()
)

display(
    pilot_df.head()
)

Result CSVs:
SMAP-L4-V008-pilot-SPL4SMGP-008-results.csv

Result shape: (560, 9)

Columns:
['Category', 'ID', 'Latitude', 'Longitude', 'Date', 'SMAP_Tile', 'SPL4SMGP_008_Line_Y_9.00805km', 'SPL4SMGP_008_Sample_X_9.00805km', 'SPL4SMGP_008_Geophysical_Data_sm_surface']


,Category,ID,Latitude,Longitude,Date,SMAP_Tile,SPL4SMGP_008_Line_Y_9.00805km,SPL4SMGP_008_Sample_X_9.00805km,SPL4SMGP_008_Geophysical_Data_sm_surface
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,0,13.5325,2.6604,2015-04-01 01:30:00 UTC,h00v00,622.0,1956.0,0.022080
1,AMMA-CATCH::AMMA-CATCH__Banizoumbou,0,13.5325,2.6604,2015-04-01 04:30:00 UTC,h00v00,622.0,1956.0,0.023196
2,AMMA-CATCH::AMMA-CATCH__Banizoumbou,0,13.5325,2.6604,2015-04-01 07:30:00 UTC,h00v00,622.0,1956.0,0.023786
3,AMMA-CATCH::AMMA-CATCH__Banizoumbou,0,13.5325,2.6604,2015-04-01 10:30:00 UTC,h00v00,622.0,1956.0,0.023841
4,AMMA-CATCH::AMMA-CATCH__Banizoumbou,0,13.5325,2.6604,2015-04-01 13:30:00 UTC,h00v00,622.0,1956.0,0.022675


In [14]:
# ============================================================
# PARSE APPEEARS SMAP 3-HOURLY OUTPUT
# ============================================================

base_columns = {
    "Category",
    "ID",
    "Latitude",
    "Longitude",
    "Date",
}


candidate_value_columns = []

for col in pilot_df.columns:

    if (
        col not in base_columns
        and (
            "sm_surface"
            in col.lower()
        )
    ):

        candidate_value_columns.append(
            col
        )

print(
    "SMAP surface columns found:",
    candidate_value_columns
)

if len(
    candidate_value_columns
) != 1:

    raise RuntimeError(
        "Could not identify exactly one "
        "SMAP sm_surface output column.\n"
        f"Found: {candidate_value_columns}"
    )

SMAP_APPEEARS_COLUMN = (
    candidate_value_columns[0]
)

print(
    "Using column:",
    SMAP_APPEEARS_COLUMN
)


# ------------------------------------------------------------
# Parse timestamps
# ------------------------------------------------------------

pilot_df[
    "Date"
] = pd.to_datetime(
    pilot_df[
        "Date"
    ],
    errors="coerce",
)

pilot_df[
    "sample_date"
] = (
    pilot_df[
        "Date"
    ]
    .dt
    .normalize()
)


# ------------------------------------------------------------
# Numeric conversion
# ------------------------------------------------------------

pilot_df[
    SMAP_APPEEARS_COLUMN
] = pd.to_numeric(
    pilot_df[
        SMAP_APPEEARS_COLUMN
    ],
    errors="coerce",
)


# ------------------------------------------------------------
# Valid SMAP values
# ------------------------------------------------------------

pilot_df.loc[
    (
        pilot_df[
            SMAP_APPEEARS_COLUMN
        ] < 0.0
    )
    |
    (
        pilot_df[
            SMAP_APPEEARS_COLUMN
        ] > 0.9
    ),
    SMAP_APPEEARS_COLUMN,
] = np.nan


# ------------------------------------------------------------
# Daily aggregation
# ------------------------------------------------------------

pilot_daily = (
    pilot_df
    .groupby(
        [
            "ID",
            "sample_date",
        ],
        as_index=False,
    )
    .agg(
        SMAP=(
            SMAP_APPEEARS_COLUMN,
            "mean",
        ),

        SMAP_n_obs=(
            SMAP_APPEEARS_COLUMN,
            "count",
        ),
    )
)

print(
    "Daily rows:",
    len(pilot_daily)
)

print(
    "Mean observations/day:",
    pilot_daily[
        "SMAP_n_obs"
    ].mean()
)

print(
    "Daily valid:",
    pilot_daily[
        "SMAP"
    ].notna().sum()
)

display(
    pilot_daily.head(
        20
    )
)

SMAP surface columns found: ['SPL4SMGP_008_Geophysical_Data_sm_surface']
Using column: SPL4SMGP_008_Geophysical_Data_sm_surface
Daily rows: 70
Mean observations/day: 8.0
Daily valid: 70


,ID,sample_date,SMAP,SMAP_n_obs
0,0,2015-04-01 00:00:00+00:00,0.022373,8
1,0,2015-04-02 00:00:00+00:00,0.021263,8
2,0,2015-04-03 00:00:00+00:00,0.023378,8
3,0,2015-04-04 00:00:00+00:00,0.023353,8
4,0,2015-04-05 00:00:00+00:00,0.023167,8
5,0,2015-04-06 00:00:00+00:00,0.026044,8
6,0,2015-04-07 00:00:00+00:00,0.023846,8
7,1,2015-04-01 00:00:00+00:00,0.140224,8
8,1,2015-04-02 00:00:00+00:00,0.116250,8
9,1,2015-04-03 00:00:00+00:00,0.123323,8


In [25]:
# ============================================================
# APPEEARS SCALE TEST — 3 POINT BATCHES
# 2,210 STATIONS × 30 DAYS
# ============================================================

SCALE_START = pd.Timestamp(
    "2020-10-04"
)

SCALE_END = pd.Timestamp(
    "2026-10-04"
)

MAX_COORDINATES = 1000


# ------------------------------------------------------------
# Load stations
# ------------------------------------------------------------

if "stations" not in globals():

    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

if "station_key" not in stations.columns:

    stations["network"] = (
        stations["network"]
        .astype(str)
    )

    stations["station_id"] = (
        stations["station_id"]
        .astype(str)
    )

    stations["station_key"] = (
        stations["network"]
        + "::"
        + stations["station_id"]
    )


# ------------------------------------------------------------
# Prepare all stations
# ------------------------------------------------------------

scale_stations = (
    stations[
        [
            "station_key",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates(
        "station_key"
    )
    .sort_values(
        "station_key"
    )
    .reset_index(
        drop=True
    )
)

scale_stations[
    "appeears_id"
] = np.arange(
    len(scale_stations)
).astype(
    str
)

print(
    "Total stations:",
    len(scale_stations)
)


# ------------------------------------------------------------
# Split into <= 1,000-coordinate batches
# ------------------------------------------------------------

scale_batches = [
    scale_stations.iloc[
        start:start + MAX_COORDINATES
    ].copy()
    for start in range(
        0,
        len(scale_stations),
        MAX_COORDINATES,
    )
]

print(
    "Number of batches:",
    len(scale_batches)
)

for i, batch in enumerate(
    scale_batches,
    start=1,
):

    print(
        f"Batch {i}:",
        len(batch),
        "stations"
    )


# ------------------------------------------------------------
# Submit each batch
# ------------------------------------------------------------

scale_task_ids = []

for batch_number, batch in enumerate(
    scale_batches,
    start=1,
):

   # ============================================================
# BUILD APPEEARS COORDINATES — SAFE DECIMAL PRECISION
# ============================================================

    coordinates = []

    for _, row in batch.iterrows():

        lat = round(float(row["latitude"]), 6)
        lon = round(float(row["longitude"]), 6)

        coordinates.append({

            "id":
                str(row["appeears_id"]),

            "latitude":
                lat,

            "longitude":
                lon,

            "category":
                str(row["station_key"]),
        })


    task = {

        "task_type":
            "point",

        "task_name":
            (
                "SMAP_L4_V008_SCALE_"
                f"2015_04_B{batch_number}"
            ),

        "params": {

            "dates": [
                {
                    "startDate":
                        SCALE_START.strftime(
                            "%m-%d-%Y"
                        ),

                    "endDate":
                        SCALE_END.strftime(
                            "%m-%d-%Y"
                        ),
                }
            ],

            "layers": [
                {
                    "product":
                        "SPL4SMGP.008",

                    "layer":
                        "Geophysical_Data_sm_surface",
                }
            ],

            "coordinates":
                coordinates,
        },
    }


    print(
        f"\nSubmitting batch {batch_number}/"
        f"{len(scale_batches)}..."
    )

    response = requests.post(
        APPEEARS_API + "task",
        json=task,
        headers=appeears_headers,
        timeout=120,
    )

    print(
        "HTTP:",
        response.status_code
    )

    print(
        response.text[:1000]
    )

    response.raise_for_status()

    task_json = response.json()

    task_id = (
        task_json[
            "task_id"
        ]
    )

    scale_task_ids.append(
        task_id
    )

    print(
        "Task ID:",
        task_id
    )


print(
    "\nSubmitted tasks:"
)

for i, task_id in enumerate(
    scale_task_ids,
    start=1,
):

    print(
        f"Batch {i}: {task_id}"
    )

Total stations: 2210
Number of batches: 3
Batch 1: 1000 stations
Batch 2: 1000 stations
Batch 3: 210 stations

Submitting batch 1/3...
HTTP: 202
{"task_id": "064e2b29-8251-43e7-ba2b-d96fe8835e09", "status": "pending"}
Task ID: 064e2b29-8251-43e7-ba2b-d96fe8835e09

Submitting batch 2/3...
HTTP: 202
{"task_id": "8bddc625-e47d-4efd-8954-c08ca88eeefd", "status": "pending"}
Task ID: 8bddc625-e47d-4efd-8954-c08ca88eeefd

Submitting batch 3/3...
HTTP: 202
{"task_id": "9344de41-9ddc-4023-aafe-a240a8462bf3", "status": "pending"}
Task ID: 9344de41-9ddc-4023-aafe-a240a8462bf3

Submitted tasks:
Batch 1: 064e2b29-8251-43e7-ba2b-d96fe8835e09
Batch 2: 8bddc625-e47d-4efd-8954-c08ca88eeefd
Batch 3: 9344de41-9ddc-4023-aafe-a240a8462bf3


In [21]:
# ============================================================
# GEE EECU BENCHMARK
# 2,210 stations × 30 days
# SPL4SMGP.008 / sm_surface
# ============================================================

import ee
import pandas as pd
import time
from datetime import datetime

# ------------------------------------------------------------
# 1. INITIALIZE EARTH ENGINE
# ------------------------------------------------------------

try:
    ee.Initialize(project='global-downscaling')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='global-downscaling')

print("Earth Engine initialized.")


# ------------------------------------------------------------
# 2. LOAD STATIONS
# ------------------------------------------------------------

if "stations" not in globals():
    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

stations_gee = (
    stations[
        [
            "station_key",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates("station_key")
    .sort_values("station_key")
    .reset_index(drop=True)
)

print(
    "Stations:",
    len(stations_gee)
)


# ------------------------------------------------------------
# 3. CONVERT STATIONS TO EE FEATURE COLLECTION
# ------------------------------------------------------------

features = []

for _, row in stations_gee.iterrows():

    features.append(
        ee.Feature(
            ee.Geometry.Point([
                float(row["longitude"]),
                float(row["latitude"]),
            ]),
            {
                "station_key":
                    str(row["station_key"]),
            },
        )
    )

station_fc = ee.FeatureCollection(
    features
)

print("Station FeatureCollection ready.")


# ------------------------------------------------------------
# 4. SMAP SPL4SMGP V008
# ------------------------------------------------------------

START_DATE = "2015-04-01"
END_DATE   = "2015-05-01"   # exclusive

smap = (
    ee.ImageCollection(
        "NASA/SMAP/SPL4SMGP/008"
    )
    .filterDate(
        START_DATE,
        END_DATE
    )
    .select("sm_surface")
)

print(
    "SMAP images:",
    smap.size().getInfo()
)


# ------------------------------------------------------------
# 5. SAMPLE EVERY 3-HOUR IMAGE AT ALL STATIONS
# ------------------------------------------------------------

def sample_image(image):

    sampled = image.sampleRegions(
        collection=station_fc,
        properties=["station_key"],
        scale=11000,
        geometries=False,
    )

    date_string = ee.Date(
        image.get("system:time_start")
    ).format(
        "YYYY-MM-dd'T'HH:mm:ss"
    )

    return sampled.map(
        lambda f: f.set(
            "timestamp",
            date_string,
            "system_time_start",
            image.get(
                "system:time_start"
            ),
        )
    )


benchmark_fc = ee.FeatureCollection(
    smap.map(sample_image)
).flatten()


# ------------------------------------------------------------
# 6. ADD A FEW COUNTERS / KEEP OUTPUT SMALL
# ------------------------------------------------------------

benchmark_fc = benchmark_fc.select(
    [
        "station_key",
        "timestamp",
        "sm_surface",
    ]
)

print(
    "Benchmark computation graph built."
)


# ------------------------------------------------------------
# 7. SUBMIT AS A BATCH EXPORT
# ------------------------------------------------------------

description = (
    "SMAP_L4_EECU_BENCHMARK_"
    "2210STATIONS_30D"
)

task = ee.batch.Export.table.toDrive(
    collection=benchmark_fc,
    description=description,
    folder="GEE_SMAP_BENCHMARK",
    fileNamePrefix=description,
    fileFormat="CSV",
)

task.start()

print(
    "\nTask submitted:"
)
print(
    task.id
)


# ------------------------------------------------------------
# 8. MONITOR ACTUAL EECU USAGE
# ------------------------------------------------------------

operation_name = (
    "projects/earthengine-legacy/operations/"
    + task.id
)

print(
    "\nMonitoring EECU usage..."
)

while True:

    op = ee.data.getOperation(
        operation_name
    )

    metadata = op.get(
        "metadata",
        {}
    )

    state = metadata.get(
        "state",
        "UNKNOWN"
    )

    eecu = metadata.get(
        "batchEecuUsageSeconds",
        0
    )

    print(
        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
        "|",
        state,
        "| EECU:",
        round(
            float(eecu),
            2
        ),
        "seconds",
    )

    if state in {
        "SUCCEEDED",
        "FAILED",
        "CANCELLED",
    }:
        break

    time.sleep(30)


# ------------------------------------------------------------
# 9. FINAL RESULT
# ------------------------------------------------------------

final_eecu = metadata.get(
    "batchEecuUsageSeconds",
    0
)

print(
    "\n========================================"
)

print(
    "FINAL STATE:",
    state
)

print(
    "FINAL EECU-SECONDS:",
    final_eecu
)

print(
    "FINAL EECU-HOURS:",
    float(final_eecu) / 3600
)

print(
    "========================================"
)

if "error" in op:
    print(
        "\nERROR:"
    )
    print(
        op["error"]
    )

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized.
Stations: 2210
Station FeatureCollection ready.
SMAP images: 240
Benchmark computation graph built.

Task submitted:
6Y6ADRAQU7S3Y6YFIVZS7LXR

Monitoring EECU usage...
2026-10-07 15:03:13 | PENDING | EECU: 0.0 seconds
2026-10-07 15:03:47 | RUNNING | EECU: 218.21 seconds
2026-10-07 15:04:18 | RUNNING | EECU: 444.94 seconds
2026-10-07 15:04:50 | RUNNING | EECU: 532.38 seconds
2026-10-07 15:05:20 | RUNNING | EECU: 535.11 seconds
2026-10-07 15:05:51 | RUNNING | EECU: 537.93 seconds
2026-10-07 15:06:21 | RUNNING | EECU: 541.34 seconds
2026-10-07 15:06:52 | RUNNING | EECU: 546.93 seconds
2026-10-07 15:07:22 | RUNNING | EECU: 551.27 seconds
2026-10-07 15:07:53 | RUNNING | EECU: 557.72 seconds
2026-10-07 15:08:24 | RUNNING | EECU: 564.9 seconds
2026-10-07 15:08:55 | RUNNING | EECU: 572.1 seconds
2026-10-07 15:09:26 | RUNNING | EECU: 578.05 seconds
2026-10-07 15:09:56 | RUNNING | EECU: 584.63 seconds
2026-10-07 15:10:26 | RUNNING | EECU: 587.19 seconds
2026-10-07 15:10

In [22]:
# ============================================================
# GEE PRODUCTION — 6 SMAP EXPORT TASKS
#
# 3 station batches:
#   B01 = 1,000 stations
#   B02 = 1,000 stations
#   B03 = remaining stations
#
# 2 temporal windows:
#   2015-04-01 -> 2020-04-01
#   2020-04-01 -> latest sample date + 1 day
#
# Output per station-day:
#   station_key
#   date
#   sm_surface = daily mean SMAP L4 soil moisture
#   smap_n     = number of valid 3-hour observations
#
# Assumes Earth Engine is ALREADY initialized.
# ============================================================

import ee
import pandas as pd


# ------------------------------------------------------------
# 1. LOAD STATIONS
# ------------------------------------------------------------

if "stations" not in globals():

    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

stations_run = (
    stations[
        [
            "station_key",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates(
        "station_key"
    )
    .sort_values(
        "station_key"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Total stations:",
    len(stations_run),
)


# ------------------------------------------------------------
# 2. LOAD SAMPLE DATES
# ------------------------------------------------------------

if "samples" not in globals():

    samples = pd.read_parquet(
        "Data/Global/pipeline_stage1/station_date_candidates.parquet"
    )

samples["sample_date"] = pd.to_datetime(
    samples["sample_date"]
)

max_sample_date = (
    samples["sample_date"]
    .max()
)

# GEE filterDate is END-EXCLUSIVE,
# so add one day to include the final date.
final_end_date = (
    max_sample_date
    + pd.Timedelta(days=1)
).strftime(
    "%Y-%m-%d"
)

print(
    "Latest sample date:",
    max_sample_date.strftime(
        "%Y-%m-%d"
    ),
)

print(
    "GEE end date:",
    final_end_date,
)


# ------------------------------------------------------------
# 3. SPLIT STATIONS INTO 3 BATCHES
# ------------------------------------------------------------

MAX_POINTS = 1000

station_batches = [
    stations_run.iloc[
        start:start + MAX_POINTS
    ].copy()
    for start in range(
        0,
        len(stations_run),
        MAX_POINTS,
    )
]

print(
    "\nStation batches:"
)

for i, batch in enumerate(
    station_batches,
    start=1,
):

    print(
        f"  B{i:02d}: "
        f"{len(batch):,} stations"
    )


# ------------------------------------------------------------
# 4. DEFINE TWO TEMPORAL WINDOWS
# ------------------------------------------------------------

DATE_WINDOWS = [

    (
        "2015-04-01",
        "2020-04-01",
        "2015_2020",
    ),

    (
        "2020-04-01",
        final_end_date,
        "2020_end",
    ),
]

print(
    "\nTemporal windows:"
)

for (
    start_date,
    end_date,
    period_name,
) in DATE_WINDOWS:

    print(
        f"  {period_name}: "
        f"{start_date} -> {end_date}"
    )


# ------------------------------------------------------------
# 5. SMAP COLLECTION
# ------------------------------------------------------------

SMAP_COLLECTION = (
    ee.ImageCollection(
        "NASA/SMAP/SPL4SMGP/008"
    )
    .select(
        "sm_surface"
    )
)

SMAP_PROJECTION = (
    ee.Image(
        SMAP_COLLECTION.first()
    ).projection()
)

print(
    "\nSMAP collection ready."
)


# ------------------------------------------------------------
# 6. START ALL 6 EXPORT TASKS
# ------------------------------------------------------------

started_tasks = []


for (
    start_date,
    end_date,
    period_name,
) in DATE_WINDOWS:

    print(
        "\n========================================"
    )

    print(
        "Building period:",
        period_name,
    )

    print(
        "Start:",
        start_date,
    )

    print(
        "End:",
        end_date,
    )

    print(
        "========================================"
    )


    # --------------------------------------------------------
    # PERIOD DATE RANGE
    # --------------------------------------------------------

    start_ee = ee.Date(
        start_date
    )

    end_ee = ee.Date(
        end_date
    )

    n_days = (
        end_ee
        .difference(
            start_ee,
            "day",
        )
    )

    day_indices = (
        ee.List.sequence(
            0,
            n_days.subtract(1),
        )
    )


    # --------------------------------------------------------
    # BUILD DAILY SMAP IMAGES
    #
    # sm_surface = daily mean of valid 3-hour observations
    # smap_n     = number of valid observations
    # --------------------------------------------------------

    def make_daily_image(
        day_index
    ):

        day = (
            start_ee
            .advance(
                ee.Number(
                    day_index
                ),
                "day",
            )
        )

        day_end = (
            day.advance(
                1,
                "day",
            )
        )

        day_collection = (
            SMAP_COLLECTION
            .filterDate(
                day,
                day_end,
            )
        )

        daily_mean = (
            day_collection
            .mean()
            .rename(
                "sm_surface"
            )
        )

        daily_count = (
            day_collection
            .count()
            .rename(
                "smap_n"
            )
        )

        return (
            daily_mean
            .addBands(
                daily_count
            )
            .set(
                "date",
                day.format(
                    "YYYY-MM-dd"
                ),
            )
            .set(
                "system:time_start",
                day.millis(),
            )
        )


    daily_images = (
        ee.ImageCollection
        .fromImages(
            day_indices.map(
                make_daily_image
            )
        )
    )

    daily_image_list = (
        daily_images.toList(
            daily_images.size()
        )
    )


    # --------------------------------------------------------
    # 7. RUN THE 3 STATION BATCHES
    # --------------------------------------------------------

    for (
        batch_number,
        batch,
    ) in enumerate(
        station_batches,
        start=1,
    ):

        print(
            "\nBuilding:",
            f"{period_name} / "
            f"B{batch_number:02d}",
        )


        # ----------------------------------------------------
        # BUILD EE FEATURE COLLECTION
        # ----------------------------------------------------

        features = []

        for _, row in batch.iterrows():

            features.append(
                ee.Feature(
                    ee.Geometry.Point(
                        [
                            float(
                                row[
                                    "longitude"
                                ]
                            ),
                            float(
                                row[
                                    "latitude"
                                ]
                            ),
                        ]
                    ),
                    {
                        "station_key":
                            str(
                                row[
                                    "station_key"
                                ]
                            ),
                    },
                )
            )

        station_fc = (
            ee.FeatureCollection(
                features
            )
        )


        # ----------------------------------------------------
        # SAMPLE EACH DAILY IMAGE
        # ----------------------------------------------------

        def sample_one_day(
            day_index
        ):

            image = ee.Image(
                daily_image_list.get(
                    day_index
                )
            )

            sampled = (
                image.sampleRegions(
                    collection=
                        station_fc,

                    properties=[
                        "station_key"
                    ],

                    projection=
                        SMAP_PROJECTION,

                    geometries=False,

                    tileScale=4,
                )
            )

            return sampled.map(
                lambda f:
                    f.set(
                        "date",
                        image.get(
                            "date"
                        ),
                    )
            )


        daily_sample_collections = (
            day_indices.map(
                sample_one_day
            )
        )


        result = (
            ee.FeatureCollection(
                daily_sample_collections
            )
            .flatten()
            .select(
                [
                    "station_key",
                    "date",
                    "sm_surface",
                    "smap_n",
                ]
            )
        )


        # ----------------------------------------------------
        # EXPORT TO GOOGLE DRIVE
        # ----------------------------------------------------

        description = (
            "SMAP_L4_DAILY_"
            f"{period_name}_"
            f"B{batch_number:02d}"
        )

        task = (
            ee.batch.Export.table.toDrive(
                collection=result,

                description=
                    description,

                folder=
                    "SMAP_GEE_PRODUCTION",

                fileNamePrefix=
                    description,

                fileFormat=
                    "CSV",
            )
        )

        task.start()


        started_tasks.append(
            {
                "description":
                    description,

                "task_id":
                    task.id,

                "period":
                    period_name,

                "batch":
                    batch_number,

                "stations":
                    len(batch),
            }
        )


        print(
            "STARTED:",
            description,
        )

        print(
            "TASK ID:",
            task.id,
        )


# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

print(
    "\n\n============================================"
)

print(
    "ALL EXPORT TASKS SUBMITTED"
)

print(
    "Total tasks:",
    len(started_tasks),
)

print(
    "============================================"
)

for t in started_tasks:

    print(
        f'{t["description"]} | '
        f'{t["stations"]:,} stations | '
        f'{t["task_id"]}'
    )

Total stations: 2210
Latest sample date: 2026-10-03
GEE end date: 2026-10-04

Station batches:
  B01: 1,000 stations
  B02: 1,000 stations
  B03: 210 stations

Temporal windows:
  2015_2020: 2015-04-01 -> 2020-04-01
  2020_end: 2020-04-01 -> 2026-10-04

SMAP collection ready.

Building period: 2015_2020
Start: 2015-04-01
End: 2020-04-01

Building: 2015_2020 / B01
STARTED: SMAP_L4_DAILY_2015_2020_B01
TASK ID: BD2P3YW3ZPHO7LFDQIPQHJJN

Building: 2015_2020 / B02
STARTED: SMAP_L4_DAILY_2015_2020_B02
TASK ID: HQS7GNDAKEQQ4ZJ6AD4JRKLS

Building: 2015_2020 / B03
STARTED: SMAP_L4_DAILY_2015_2020_B03
TASK ID: JY5FSWBDOCDRPZJSDPI2M3IF

Building period: 2020_end
Start: 2020-04-01
End: 2026-10-04

Building: 2020_end / B01
STARTED: SMAP_L4_DAILY_2020_end_B01
TASK ID: QGEMJ77HMW47KXW7AULK7WG3

Building: 2020_end / B02
STARTED: SMAP_L4_DAILY_2020_end_B02
TASK ID: CF4GBBAMCOJQGV3TMNWWVFFG

Building: 2020_end / B03
STARTED: SMAP_L4_DAILY_2020_end_B03
TASK ID: X6TB2A6NTV4NY4MBMVSHWSHW


ALL EXPORT TASKS

In [19]:
# ============================================================
# WAIT FOR ALL SCALE-TEST TASKS
# ============================================================

POLL_SECONDS = 30

remaining = {
    task_id
    for task_id in scale_task_ids
}

scale_status = {}

while remaining:

    print(
        "\nChecking tasks..."
    )

    finished_now = set()

    for task_id in list(
        remaining
    ):

        response = requests.get(
            APPEEARS_API
            + "task/"
            + task_id,
            headers=appeears_headers,
            timeout=60,
        )

        response.raise_for_status()

        info = response.json()

        status = info[
            "status"
        ]

        scale_status[
            task_id
        ] = info

        print(
            task_id,
            "|",
            status,
        )

        if status in {
            "done",
            "failed",
            "error",
        }:

            finished_now.add(
                task_id
            )

    remaining -= finished_now

    if remaining:

        time.sleep(
            POLL_SECONDS
        )


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

failed = [
    task_id
    for task_id, info
    in scale_status.items()
    if info["status"] != "done"
]

print(
    "\nFinished tasks:",
    len(scale_status)
)

print(
    "Successful:",
    len(scale_status) - len(failed)
)

print(
    "Failed:",
    len(failed)
)

if failed:

    for task_id in failed:

        print(
            "\nFAILED:",
            task_id
        )

        print(
            scale_status[
                task_id
            ]
        )

    raise RuntimeError(
        "One or more AppEEARS scale-test tasks failed."
    )

print(
    "\nAll AppEEARS scale-test tasks completed successfully."
)


Checking tasks...
f9128bc3-3a47-4c96-97a3-87d7028ed879 | queued
b095a870-ee5b-4c70-8c53-0d103ebf6000 | pending
577ab539-7180-491d-b0ba-85dd3b75a826 | pending

Checking tasks...
f9128bc3-3a47-4c96-97a3-87d7028ed879 | processing
b095a870-ee5b-4c70-8c53-0d103ebf6000 | queued
577ab539-7180-491d-b0ba-85dd3b75a826 | processing

Checking tasks...
f9128bc3-3a47-4c96-97a3-87d7028ed879 | processing
b095a870-ee5b-4c70-8c53-0d103ebf6000 | processing
577ab539-7180-491d-b0ba-85dd3b75a826 | processing

Checking tasks...
f9128bc3-3a47-4c96-97a3-87d7028ed879 | processing
b095a870-ee5b-4c70-8c53-0d103ebf6000 | processing
577ab539-7180-491d-b0ba-85dd3b75a826 | processing

Checking tasks...
f9128bc3-3a47-4c96-97a3-87d7028ed879 | processing
b095a870-ee5b-4c70-8c53-0d103ebf6000 | processing
577ab539-7180-491d-b0ba-85dd3b75a826 | processing

Checking tasks...
f9128bc3-3a47-4c96-97a3-87d7028ed879 | processing
b095a870-ee5b-4c70-8c53-0d103ebf6000 | processing
577ab539-7180-491d-b0ba-85dd3b75a826 | processing


SSLError: HTTPSConnectionPool(host='appeears.earthdatacloud.nasa.gov', port=443): Max retries exceeded with url: /api/task/577ab539-7180-491d-b0ba-85dd3b75a826 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1081)')))